# Clasificación de series temporales FordA

[Abrir y ejecutar en Google Colab](https://colab.research.google.com/github/al073016/Practica-4-Modelos-de-machine-learning/blob/main/03_Series_Tiempo_FordA.ipynb)

Pregunta: ¿qué clase corresponde a una señal de ruido de motor de 500 mediciones? Fuente: [FordA, archivo UCR/UEA](https://www.timeseriesclassification.com/description.php?Dataset=FordA); la [guía oficial de Keras](https://keras.io/examples/timeseries/timeseries_classification_from_scratch/) describe la descarga y la naturaleza de las señales. Se respetan los archivos originales TRAIN y TEST. **Importante:** FordA codifica las clases como −1 y 1, pero no publica aquí una equivalencia fiable entre cada etiqueta y «normal/falla»; se muestran como clase 0 y clase 1 sin inventar su significado.

**Uso:** abre este notebook en Google Colab y elige **Entorno de ejecución → Ejecutar todo**. Las celdas descargan los datos automáticamente. La última celda permite probar una muestra del conjunto de prueba. El entrenamiento en CPU puede tardar varios minutos.

## 1. Bibliotecas y descarga automática

In [ ]:
import importlib.util, subprocess, sys
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
          'sklearn': 'scikit-learn', 'tensorflow': 'tensorflow', 'openpyxl': 'openpyxl'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
import urllib.request
from sklearn.metrics import f1_score, accuracy_score, classification_report

root = 'https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/'
train_raw = np.loadtxt(root + 'FordA_TRAIN.tsv', delimiter='\t', dtype='float32')
test_raw = np.loadtxt(root + 'FordA_TEST.tsv', delimiter='\t', dtype='float32')
X_dev = train_raw[:, 1:, None]
y_dev = (train_raw[:, 0] == 1).astype('int32')
X_test = test_raw[:, 1:, None]
y_test = (test_raw[:, 0] == 1).astype('int32')
assert X_dev.shape == (3601,500,1) and X_test.shape == (1320,500,1)
print('Entrenamiento original:', X_dev.shape, 'Prueba original:', X_test.shape)
print('Clases en entrenamiento:', np.bincount(y_dev))
for label in (0,1):
    plt.plot(X_dev[np.where(y_dev == label)[0][0], :, 0], label=f'Clase {label}')
plt.title('Ejemplo de cada clase'); plt.legend(); plt.show()

## 2. Validación y arquitectura
Se reserva 20 % del archivo TRAIN para validación. FordA ya viene normalizado por serie. Se comparan dos tamaños de CNN 1D; el archivo TEST se utiliza solo al final.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.20, stratify=y_dev, random_state=SEED)

def build_model(filters):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(500,1)),
        tf.keras.layers.Conv1D(filters[0], 5, padding='same'),
        tf.keras.layers.BatchNormalization(), tf.keras.layers.ReLU(),
        tf.keras.layers.Conv1D(filters[1], 5, padding='same'),
        tf.keras.layers.BatchNormalization(), tf.keras.layers.ReLU(),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

trials = []
for filters in [(32,64), (64,64)]:
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(filters)
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=100, batch_size=64, verbose=0,
                        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
                                   tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.5)])
    p_val = model.predict(X_val, verbose=0).ravel()
    score = f1_score(y_val, p_val >= 0.5)
    trials.append((score, filters, model, history))
    print(filters, 'épocas:', len(history.history['loss']), 'F1 validación:', round(score,4))

best_f1, best_filters, model, history = max(trials, key=lambda t: t[0])
print('Modelo elegido:', best_filters)

## 3. Evaluación final en prueba

In [ ]:
p_test = model.predict(X_test, verbose=0).ravel()
y_pred = (p_test >= 0.5).astype('int32')
print('Accuracy:', round(accuracy_score(y_test, y_pred),4), 'F1:', round(f1_score(y_test, y_pred),4))
print('Referencia accuracy por clase mayoritaria:', round(float(np.bincount(y_test).max() / len(y_test)),4))
print(classification_report(y_test, y_pred, target_names=['Clase 0 (-1)','Clase 1 (+1)']))
fig, axes = plt.subplots(1, 2, figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], colorbar=False)
axes[1].plot(history.history['loss'], label='Entrenamiento')
axes[1].plot(history.history['val_loss'], label='Validación')
axes[1].set_title('Pérdida'); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Probar una señal
Modifica `indice` para explorar otra señal del conjunto de prueba.

In [ ]:
indice = 0
assert 0 <= indice < len(X_test)
plt.plot(X_test[indice,:,0]); plt.title(f'Señal de prueba {indice}'); plt.show()
print('Probabilidad clase 1:', round(float(p_test[indice]),3), '| Predicción:', int(y_pred[indice]), '| Real:', int(y_test[indice]))